# 👑 Live Tương Tác Game Vua Tiếng Việt Lên YouTube
Notebook này hỗ trợ cả 2 lựa chọn:
1. **LỰA CHỌN 1 (Dùng OBS):** Chạy Server Game + Tạo link Cloudflare Tunnel để dán vào OBS trên máy tính.
2. **LỰA CHỌN 2 (Live Ngầm Không Cần OBS):** Máy ảo Colab tự mở màn hình ảo và dùng FFmpeg stream trực tiếp lên YouTube.

## ⚙️ Bước 0: Tải mã nguồn & cài đặt thư viện

In [ ]:
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/TikTok-Live-Interactive-Game 2>/dev/null || (cd /content/TikTok-Live-Interactive-Game && git pull)
%cd /content/TikTok-Live-Interactive-Game/games/vua-tieng-viet
!npm install
!npm install youtube-chat
!apt-get update -qq && apt-get install -y -qq xvfb ffmpeg pulseaudio >/dev/null 2>&1
print('✅ Đã cài đặt xong môi trường!')

---
## 🅰️ LỰA CHỌN 1: Phát bằng OBS Studio (Chuẩn & Đẹp nhất)
Chạy ô dưới đây để lấy **Link Game** dán vào Browser Source trong OBS:

In [ ]:
import os, subprocess, time, re, urllib.request, sys

YOUTUBE_LIVE_ID = "TuH8JlNLlxI" # @param {type:"string"}

# 1. Đảm bảo mã nguồn đã tải về
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/TikTok-Live-Interactive-Game 2>/dev/null || (cd /content/TikTok-Live-Interactive-Game && git pull)
%cd /content/TikTok-Live-Interactive-Game/games/vua-tieng-viet

# 2. Kiểm tra thư viện
if not os.path.exists('node_modules/youtube-chat'):
    print('📦 Đang cài đặt thư viện cần thiết...')
    !npm install --silent
    !npm install youtube-chat --silent

# 3. Tải Cloudflare Tunnel nếu chưa có
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('📥 Đang tải Cloudflare Tunnel...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/tmp/cloudflared')
    os.chmod('/tmp/cloudflared', 0o755)
    !mv /tmp/cloudflared /usr/local/bin/cloudflared

# 4. Dọn dẹp tiến trình cũ
!pkill -9 -f server-youtube.js >/dev/null 2>&1
!pkill -9 -f cloudflared >/dev/null 2>&1
time.sleep(1)

# 5. Khởi động server game
print('🚀 Đang khởi động Server Game kết nối YouTube Live...')
env = os.environ.copy()
env['PORT'] = '3002'
env['YOUTUBE_LIVE_ID'] = YOUTUBE_LIVE_ID

server_log = open('/tmp/server_yt.log', 'w')
server_proc = subprocess.Popen(['node', 'server-youtube.js'], env=env, stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(3)

# 6. Khởi động Cloudflare Tunnel (ghi cả stdout và stderr)
print('🌐 Đang tạo đường dẫn HTTPS công khai cho OBS...')
cf_log = open('/tmp/cf.log', 'w')
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:3002'], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for i in range(25):
    time.sleep(1)
    if os.path.exists('/tmp/cf.log'):
        with open('/tmp/cf.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

print('
' + '='*70)
if public_url:
    print(f'🎉 KẾT NỐI THÀNH CÔNG!')
    print(f'👉 [1] Link Màn Hình Game (Dán vào Browser Source trong OBS):
       {public_url}')
    print(f'👉 [2] Link Bảng Điều Khiển MC (Mở trên điện thoại/máy tính):
       {public_url}/admin.html')
else:
    print('⚠️ Chưa lấy được link. Vui lòng xem log bên dưới:')
    if os.path.exists('/tmp/cf.log'):
        with open('/tmp/cf.log', 'r') as f: print(f.read()[-500:])
print('='*70)


---
## 🅱️ LỰA CHỌN 2: Phát Live Ngầm Trực Tiếp (Không Cần OBS)
Dán **Stream Key** lấy từ YouTube Studio vào ô bên dưới rồi bấm chạy. Máy ảo sẽ tự quay màn hình và stream lên YouTube:

In [ ]:
# @title Nhập thông tin luồng Live YouTube
YOUTUBE_LIVE_ID = "TuH8JlNLlxI" # @param {type:"string"}
YOUTUBE_STREAM_KEY = "" # @param {type:"string"}

%cd /content/TikTok-Live-Interactive-Game/games/vua-tieng-viet
!python3 live_ngam_youtube.py --stream-key "$YOUTUBE_STREAM_KEY" --live-id "$YOUTUBE_LIVE_ID"